In [1]:
import csv
import pandas as pd
import numpy as np
from collections import defaultdict

In [2]:
base = "CME"

threshold = 3.5

attn = 0.1

file_CME = "Input Data/detections_CME.csv"
file_Dino = f"Input Data/ap2_attn_{attn}/ap2_attn_{attn}_centroids.csv"
df = pd.read_csv(file_Dino)
df

,timepoint,particle_id,intensity,x,y,z
0,0,0,-7.300503e+09,93.955540,342.527100,498.48660
1,0,1,2.888375e+02,0.000000,14.555616,261.02164
2,0,2,2.574749e+03,0.482249,30.700237,259.63040
3,0,3,2.237380e+04,1.987546,63.178318,261.34433
4,0,4,1.399809e+03,0.168053,92.856926,253.56152
...,...,...,...,...,...,...
160092,98,1012,1.270322e+03,166.575550,678.799300,577.74884
160093,98,1013,1.750465e+04,164.890900,687.144350,571.27167
160094,98,1014,3.727564e+03,165.662350,694.662350,518.13870
160095,98,1015,2.911209e+03,166.364380,701.122440,571.78670


In [3]:
df_CME = pd.read_csv(file_CME, header = None)
#df_CME = df_CME[df_CME[0]<=97]
display(df_CME)
M_CME = df_CME.to_numpy()
M_CME[:, 4] = M_CME[:, 4]-1

df_Dino = pd.read_csv(file_Dino, names = ["t","ID", "FI", "z", "y", "x"],skiprows=1)
df_Dino = df_Dino[["t","ID","x","y","z","FI"]]
display(df_Dino)

M_Dino = df_Dino.to_numpy()
M_Dino[:, 0] += 1

if base == "CME":
    comp_list = np.zeros((len(M_CME),16), dtype=object)
    M_x = M_CME
    M_y = M_Dino
    sec = "Dino"
if base == "Dino":
    comp_list = np.zeros((len(M_Dino),16), dtype=object)
    M_x = M_Dino
    M_y = M_CME
    sec = "CME"

,0,1,2,3,4,5
0,1,1,268.92,332.29,3.7825,74.177
1,1,2,345.96,502.36,5.4485,129.010
2,1,3,381.83,352.82,4.1066,113.810
3,1,4,386.89,375.52,4.1879,49.730
4,1,5,269.17,18.51,3.8182,139.690
...,...,...,...,...,...,...
150050,99,1469,609.07,383.28,163.6700,63.429
150051,99,1470,754.66,253.81,164.4900,54.131
150052,99,1471,508.86,622.38,164.1500,306.990
150053,99,1472,523.25,112.88,162.1600,43.188


,t,ID,x,y,z,FI
0,0,0,498.48660,342.527100,93.955540,-7.300503e+09
1,0,1,261.02164,14.555616,0.000000,2.888375e+02
2,0,2,259.63040,30.700237,0.482249,2.574749e+03
3,0,3,261.34433,63.178318,1.987546,2.237380e+04
4,0,4,253.56152,92.856926,0.168053,1.399809e+03
...,...,...,...,...,...,...
160092,98,1012,577.74884,678.799300,166.575550,1.270322e+03
160093,98,1013,571.27167,687.144350,164.890900,1.750465e+04
160094,98,1014,518.13870,694.662350,165.662350,3.727564e+03
160095,98,1015,571.78670,701.122440,166.364380,2.911209e+03


In [4]:
t_vec = defaultdict(list)
for vec_y in M_y:
    t_vec[vec_y[0]].append(vec_y)

for t_val in t_vec:
    t_vec[t_val] = np.array(t_vec[t_val])

for i, vec_x in enumerate(M_x):
    t_val = vec_x[0]
    y_group = t_vec[t_val]
    diffs = y_group[:, 2:5] - vec_x[2:5]
    dists = np.linalg.norm(diffs, axis=1)
    min_idx = np.argmin(dists)
    min_dist = dists[min_idx]
    best_vec_y = y_group[min_idx]
    comp_list[i, 0:6] = vec_x
    comp_list[i, 6:12] = best_vec_y
    comp_list[i, 12] = min_dist

    below = np.where(dists < threshold)[0]
    if len(below) > 0:
        y_ids = ",".join(str(int(x)) for x in y_group[below, 1])
        y_FI = ",".join(str(int(x)) for x in y_group[below, 5])
        dists_str = ",".join(f"{dists[j]:.2f}" for j in below)
    else:
        y_ids = ""
        y_FI = ""
        dists_str = ""
    comp_list[i, 13] = y_ids
    comp_list[i, 14] = dists_str
    comp_list[i, 15] = y_FI

In [5]:
df_comp = pd.DataFrame(comp_list, columns = ["t",f"ID ({base})", f"x ({base})", f"y ({base})", 
                                             f"z ({base})", f"FI ({base})","t_ig", f"ID ({sec})", 
                                             f"x ({sec})", f"y ({sec})", f"z ({sec})", f"FI ({sec})",
                                             "Distance",f"Multi ID ({sec})", f"Multi Distance ({sec})", f"Multi FI ({sec})"])

df_comp = df_comp.drop("t_ig", axis=1)
df_sorted = df_comp.sort_values(by=["t",f"ID ({base})"], ascending=[True, True])

In [6]:
#df_sorted.to_csv(f"Base Output/DetectionComparison.csv", index=False)
df_sorted.to_csv(f"DC_attn{attn}.csv", index=False)

In [7]:
df_sorted

,t,ID (Dino),x (Dino),y (Dino),z (Dino),FI (Dino),ID (CME),x (CME),y (CME),z (CME),FI (CME),Distance,Multi ID (CME),Multi Distance (CME),Multi FI (CME)
0,1.0,0.0,498.4866,342.5271,93.95554,-7300503000.0,671.0,463.04,341.3,77.785,242.75,38.980169,,,
1,1.0,1.0,261.02164,14.555616,0.0,288.83752,5.0,269.17,18.51,2.8182,139.69,9.485524,,,
2,1.0,2.0,259.6304,30.700237,0.482249,2574.7488,40.0,271.78,29.509,5.4171,117.79,13.167558,,,
3,1.0,3.0,261.34433,63.178318,1.987546,22373.8,16.0,269.82,49.481,4.2725,242.75,16.26882,,,
4,1.0,4.0,253.56152,92.856926,0.168053,1399.8091,15.0,267.71,106.98,3.8633,406.3,20.329672,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
160092,99.0,1012.0,577.74884,678.7993,166.57555,1270.3218,1452.0,563.26,682.46,162.78,262.97,15.418606,,,
160093,99.0,1013.0,571.27167,687.14435,164.8909,17504.646,1452.0,563.26,682.46,162.78,262.97,9.517662,,,
160094,99.0,1014.0,518.1387,694.66235,165.66235,3727.564,1433.0,504.29,688.71,159.76,247.91,16.188104,,,
160095,99.0,1015.0,571.7867,701.12244,166.36438,2911.2095,1458.0,559.0,696.93,162.96,149.18,13.88042,,,
